In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [ ]:
data_path = "gs://path-protectors-datalake/object-detection/raw/Ayyappa_Temple_FIX_1/Ayyappa_Temple_FIX_1_time_2024-05-15merged_file.csv"

In [ ]:
data = pd.read_csv(data_path)

In [ ]:
data.drop(columns=["Unnamed: 0"], axis=1, inplace=True)

In [ ]:
data = data[data["Turning Pattern"] == "Going Down"]

In [ ]:
df = data.copy()

In [ ]:
# Extract the vehicle counts for each type
vehicle_types = ['Bicycle', 'Bus', 'Cars', 'Two-Wheeler', 'Three-Wheeler', 'LCV', 'Truck']
vehicle_counts = df[vehicle_types]

# Convert the data to a NumPy array
data_numpy = vehicle_counts.values

# Split the data into training and testing sets
train_size = int(len(data_numpy) * 0.7)
train, test = data_numpy[:train_size], data_numpy[train_size:]

# Function to create sequences for LSTM
def create_sequences(input_data, seq_length, prediction_horizon=15):
    X = []
    y = []
    for i in range(len(input_data) - seq_length - prediction_horizon):
        X.append(data_numpy[i:(i + seq_length)])
        y.append(data_numpy[i + seq_length + prediction_horizon])
    return np.array(X), np.array(y)


seq_length = 15  # Adjust sequence length as needed
prediction_horizon = 30 # used for predicting future data
X_train, y_train = create_sequences(train, seq_length, prediction_horizon)
X_test, y_test = create_sequences(test, seq_length, prediction_horizon)

In [ ]:
# Create the LSTM model
# model = Sequential()
# model.add(LSTM(units=64, activation='relu', input_shape=(seq_length, 7)))  # 7 for 7 vehicle types
# model.add(Dense(7, activation='linear'))  # 7 output neurons for each vehicle type

model = Sequential()
model.add(LSTM(units=128, activation='relu', return_sequences=True, input_shape=(seq_length, 7)))
model.add(Dropout(0.2))
model.add(LSTM(units=64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(units=7, activation='linear'))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
model.fit(X_train, y_train, epochs=128, batch_size=32, validation_data=(X_test, y_test))

In [ ]:
# Make predictions for the next 15 minutes
last_15_minutes = data_numpy[-seq_length:]
X_predict = last_15_minutes.reshape(1, seq_length, 7)
predicted_vehicle_counts = model.predict(X_predict)

print("Predicted vehicle counts for the next 15 minutes:")
print(predicted_vehicle_counts)

In [182]:
import pandas as pd
import numpy as np
from pandas.core.groupby import DataFrameGroupBy
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

In [243]:
data_path = "gs://path-protectors-datalake/object-detection/raw/Ayyappa_Temple_FIX_1/Ayyappa_Temple_FIX_1_time_2024-05-15merged_file.csv"

data = pd.read_csv(data_path)

data.drop(columns=["Unnamed: 0"], axis=1, inplace=True)


In [184]:
encoder = OneHotEncoder()

# Fit the encoder to the columns to be encoded
encoder.fit(data[['Turning Pattern', 'Camera Name']])

# Transform the data using the encoder
encoded_data_array = encoder.transform(data[['Turning Pattern', 'Camera Name']]).toarray()

# Create a new DataFrame with the encoded data
encoded_dataframe = pd.DataFrame(encoded_data_array, columns=encoder.get_feature_names_out()) 


# Concatenate the encoded data with the original DataFrame (if needed)
merged_dataframe = pd.concat([data, encoded_dataframe], axis=1)


In [244]:
data.index = pd.to_datetime(data["Timestamp"])

In [245]:
data["Turning Pattern"].unique()

array(['Going Down', 'Going Up'], dtype=object)

In [186]:
# Drop the non-required columns
# merged_dataframe.drop(columns=["Turning Pattern", "Camera Name", "Timestamp"], axis=0, inplace=True)

In [246]:
goind_down_df = data[data["Turning Pattern"] == "Going Down"]

In [238]:
goind_down_df.index[16:]

DatetimeIndex(['2024-05-15 07:45:00', '2024-05-15 07:46:00',
               '2024-05-15 07:47:00', '2024-05-15 07:48:00',
               '2024-05-15 07:49:00', '2024-05-15 07:50:00',
               '2024-05-15 07:51:00', '2024-05-15 07:52:00',
               '2024-05-15 07:53:00', '2024-05-15 07:54:00',
               ...
               '2024-05-15 10:20:00', '2024-05-15 10:21:00',
               '2024-05-15 10:22:00', '2024-05-15 10:23:00',
               '2024-05-15 10:24:00', '2024-05-15 10:25:00',
               '2024-05-15 10:26:00', '2024-05-15 10:27:00',
               '2024-05-15 10:28:00', '2024-05-15 10:29:00'],
              dtype='datetime64[ns]', name='Timestamp', length=168, freq=None)

In [248]:
training_dataset = goind_down_df[:154]
evaluation_dataset = goind_down_df[154:]

cars_training_dataset = training_dataset[["Cars"]]
cars_evaluation_dataset = evaluation_dataset[["Cars"]]

In [249]:
def create_sequences(data, time_window):
    print(f"length of data: {len(data)}")
    X, y = [], []
    for i in range(len(data) - time_window):
        X.append(data[i:i + time_window])
        y.append(data[i + time_window])
    return np.array(X), np.array(y)

In [250]:
# Build the enhanced LSTM model
def build_model(input_shape):
    model = Sequential()
    model.add(LSTM(units=128, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(LSTM(units=128))
    model.add(Dropout(0.2))
    model.add(Dense(units=64, activation='relu'))
    model.add(Dense(units=input_shape[1], activation='linear'))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

In [251]:
# Function to make predictions for the next 30 minutes
def make_predictions(model, last_data, scaler, steps=15):
    predictions = []
    current_data = last_data.copy()
    for _ in range(steps):
        prediction = model.predict(np.array([current_data]))
        predictions.append(prediction[0])
        current_data = np.vstack([current_data[1:], prediction])
    predictions = scaler.inverse_transform(predictions)
    return predictions

In [252]:
# scaler = MinMaxScaler()
# scaled_dataframe = scaler.fit_transform(cars_training_dataset)
# type(scaled_dataframe)

In [253]:
# Define the time window
time_window = 30  # 30 minutes

In [254]:
# Create sequences for both directions
X_seq, y_seq = create_sequences(cars_training_dataset.to_numpy(), time_window)

length of data: 154


In [199]:
# Split the data into training and testing sets
train_size = int(len(X_seq) * 0.8)

In [200]:
X_train, X_test = X_seq[:train_size], X_seq[train_size:]

In [201]:
y_train, y_test = y_seq[:train_size], y_seq[train_size:]

In [220]:
X_seq.shape

(124, 30, 1)

In [255]:
model = build_model((X_seq.shape[1], X_seq.shape[2]))

In [256]:
model.summary()

Model: "sequential_12"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_23 (LSTM)              (None, 30, 128)           66560     
                                                                 
 dropout_22 (Dropout)        (None, 30, 128)           0         
                                                                 
 lstm_24 (LSTM)              (None, 128)               131584    
                                                                 
 dropout_23 (Dropout)        (None, 128)               0         
                                                                 
 dense_20 (Dense)            (None, 64)                8256      
                                                                 
 dense_21 (Dense)            (None, 1)                 65        
                                                                 
Total params: 206,465
Trainable params: 206,465
Non-t

In [257]:
model.fit(X_seq, y_seq, epochs=128, batch_size=32)

Epoch 1/128
4/4 [==============================] - 6s 73ms/step - loss: 85.9006
Epoch 2/128
4/4 [==============================] - 0s 70ms/step - loss: 47.8301
Epoch 3/128
4/4 [==============================] - 0s 69ms/step - loss: 36.8973
Epoch 4/128
4/4 [==============================] - 0s 71ms/step - loss: 35.9874
Epoch 5/128
4/4 [==============================] - 0s 71ms/step - loss: 35.5123
Epoch 6/128
4/4 [==============================] - 0s 70ms/step - loss: 35.1294
Epoch 7/128
4/4 [==============================] - 0s 71ms/step - loss: 34.2064
Epoch 8/128
4/4 [==============================] - 0s 69ms/step - loss: 34.3930
Epoch 9/128
4/4 [==============================] - 0s 69ms/step - loss: 34.1473
Epoch 10/128
4/4 [==============================] - 0s 69ms/step - loss: 34.5530
Epoch 11/128
4/4 [==============================] - 0s 71ms/step - loss: 34.3823
Epoch 12/128
4/4 [==============================] - 0s 68ms/step - loss: 33.9748
Epoch 13/128
4/4 [===================

In [258]:
predictions = list()
last_30mins_data = cars_evaluation_dataset.to_numpy()

In [259]:
for _ in range(30):
    prediction = model.predict(np.array([last_30mins_data]))
    print(prediction)
    predictions.append(prediction[0])
    last_30mins_data = np.vstack([last_30mins_data[1:], prediction])

1/1 [==============================] - 2s 2s/step
[[10.900756]]
1/1 [==============================] - 1s 1s/step
[[12.587845]]
1/1 [==============================] - 0s 35ms/step
[[10.862387]]
1/1 [==============================] - 0s 32ms/step
[[9.141856]]
1/1 [==============================] - 0s 35ms/step
[[10.965791]]
1/1 [==============================] - 0s 38ms/step
[[16.368649]]
1/1 [==============================] - 0s 34ms/step
[[17.755064]]
1/1 [==============================] - 0s 32ms/step
[[16.014587]]
1/1 [==============================] - 0s 37ms/step
[[14.247458]]
1/1 [==============================] - 0s 36ms/step
[[15.477805]]
1/1 [==============================] - 0s 36ms/step
[[17.623173]]
1/1 [==============================] - 0s 33ms/step
[[17.151627]]
1/1 [==============================] - 0s 35ms/step
[[14.56179]]
1/1 [==============================] - 0s 36ms/step
[[12.524181]]
1/1 [==============================] - 0s 34ms/step
[[12.077546]]
1/1 [===========

In [210]:
predictions = scaler.inverse_transform(predictions)

In [226]:
predicted_data = pd.DataFrame(predictions, columns=cars_evaluation_dataset.columns)

In [228]:
predicted_data['Cars'] = predicted_data['Cars'].apply(lambda count: max(0, int(count)))


In [230]:
predicted_data.sum()

Cars    10
dtype: int64

In [232]:
d = {
    "car": "oCar",
    "motorcycle": "Motorcycle",
    "bus": "oBus",
    "IndianBicycle": "Bicycle",
    "IndianBus": "Bus",
    "IndianCar": "Cars",
    "Two-Wheeler": "Two-Wheeler",
    "Three-Wheeler": "Three-Wheeler",
    "LCV": "LCV",
    "IndianTruck": "Truck"
}

In [233]:
d.values()

dict_values(['oCar', 'Motorcycle', 'oBus', 'Bicycle', 'Bus', 'Cars', 'Two-Wheeler', 'Three-Wheeler', 'LCV', 'Truck'])

In [242]:
!python prediction.py

2024-08-25 12:08:26.579743: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-25 12:08:27.697715: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64
2024-08-25 12:08:27.697837: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/loca

In [379]:
import logging
from collections import defaultdict
import pandas as pd
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [393]:
def create_sequences(data, window):
    print(f"length of data: {len(data)}")
    x, y = [], []
    for index in range(len(data) - window):
        x.append(data[index:index + window])
        y.append(data[index + window])
    return np.array(x), np.array(y)


def build_model(input_shape):
    """build the LSTM model with 2 input, dense and dropout layers each"""
    model = Sequential()
    model.add(LSTM(units=64, return_sequences=True, input_shape=input_shape))
    model.add(LSTM(units=64))
    model.add(Dense(units=32, activation="relu"))
    model.add(Dense(units=input_shape[1], activation="linear"))
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model


def predict(model, actual_data, step_window=15, time_window=15) -> int:
    """function to make predictions for the next 15 minutes"""
    predictions = []
    actual_data_ndarray = actual_data.to_numpy()
    actual_data_ndarray = actual_data_ndarray[-step_window:]
    for _ in range(time_window):
        prediction = model.predict(np.array([actual_data_ndarray]))
        print(f"prediction: {prediction[0]}")
        predictions.append(prediction[0])
        actual_data_ndarray = np.vstack([actual_data_ndarray[1:], prediction])
    predicted_dataframe = pd.DataFrame(predictions, columns=actual_data.columns)
    predicted_dataframe.to_csv("predicted_dataframe_orginal.csv")
    predicted_dataframe = predicted_dataframe.applymap(lambda count: max(0, int(count)))
    predicted_dataframe.to_csv("predicted_dataframe_rounded.csv")
    return predicted_dataframe.sum()

In [381]:
data_path = "merged_file.csv"
data = pd.read_csv(data_path)
data.drop(columns=["Unnamed: 0"], axis=1, inplace=True)

In [382]:
data.index = pd.to_datetime(data["Timestamp"])

    # find all the unique turning patterns
turning_patterns = data["Turning Pattern"].unique()

    # Define the window sequence size
window_size = 25

    # datastore for storing predictions
predicted_datastore = defaultdict(lambda: dict())

In [383]:
turning_pattern_dataframe = data[data["Turning Pattern"] == "Going Down"]

In [384]:
target_vehicle_class = ["Bicycle", "Bus", "Cars", "Two-Wheeler", "Three-Wheeler", "LCV", "Truck"]

In [385]:
vehicle_turning_pattern_dataframe = turning_pattern_dataframe[target_vehicle_class]

In [386]:
x_sequence, y_sequence = create_sequences(vehicle_turning_pattern_dataframe.to_numpy(), window_size)

length of data: 30


In [387]:
y_sequence.shape

(5, 7)

In [388]:
model = build_model((x_sequence.shape[1], x_sequence.shape[2]))

In [389]:
model.summary()

Model: "sequential_19"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_37 (LSTM)              (None, 25, 64)            18432     
                                                                 
 lstm_38 (LSTM)              (None, 64)                33024     
                                                                 
 dense_34 (Dense)            (None, 32)                2080      
                                                                 
 dense_35 (Dense)            (None, 7)                 231       
                                                                 
Total params: 53,767
Trainable params: 53,767
Non-trainable params: 0
_________________________________________________________________


In [390]:
model.fit(x_sequence, y_sequence, epochs=64, batch_size=32)

Epoch 1/64
1/1 [==============================] - 4s 4s/step - loss: 23.1235
Epoch 2/64
1/1 [==============================] - 0s 29ms/step - loss: 22.1275
Epoch 3/64
1/1 [==============================] - 0s 29ms/step - loss: 21.8079
Epoch 4/64
1/1 [==============================] - 0s 30ms/step - loss: 21.4480
Epoch 5/64
1/1 [==============================] - 0s 28ms/step - loss: 21.0818
Epoch 6/64
1/1 [==============================] - 0s 27ms/step - loss: 20.7131
Epoch 7/64
1/1 [==============================] - 0s 26ms/step - loss: 20.3376
Epoch 8/64
1/1 [==============================] - 0s 26ms/step - loss: 19.9630
Epoch 9/64
1/1 [==============================] - 0s 26ms/step - loss: 19.6018
Epoch 10/64
1/1 [==============================] - 0s 26ms/step - loss: 19.2475
Epoch 11/64
1/1 [==============================] - 0s 27ms/step - loss: 18.9157
Epoch 12/64
1/1 [==============================] - 0s 26ms/step - loss: 18.5868
Epoch 13/64
1/1 [==============================] - 

In [394]:
predicted_vehicle_count = predict(model, vehicle_turning_pattern_dataframe, step_window=window_size, time_window=30)

1/1 [==============================] - 0s 28ms/step
prediction: [-1.2805183  0.6994804  4.4942837  6.864035   4.3876195 -0.2783233
 -2.4551895]
1/1 [==============================] - 0s 28ms/step
prediction: [-1.2979769  0.7049807  4.5155697  6.880915   4.405577  -0.2850772
 -2.4507225]
1/1 [==============================] - 0s 26ms/step
prediction: [-1.2977512   0.7050554   4.5160275   6.883149    4.406643   -0.28472072
 -2.4519992 ]
1/1 [==============================] - 0s 27ms/step
prediction: [-1.297714    0.70515996  4.516253    6.8838344   4.40706    -0.2846732
 -2.4523036 ]
1/1 [==============================] - 0s 26ms/step
prediction: [-1.2975538   0.70521194  4.5162287   6.8840804   4.407174   -0.28460813
 -2.4524956 ]
1/1 [==============================] - 0s 25ms/step
prediction: [-1.2974117   0.70523995  4.516158    6.8841968   4.407212   -0.2845568
 -2.4526284 ]
1/1 [==============================] - 0s 27ms/step
prediction: [-1.2973018   0.70525295  4.516073    6.884236